# 🧩 8-Puzzle Game Solver using A* Search

Educational notebook version of the project.

## Imports, Goal State, and Board Representation

In [ ]:
import tkinter as tkinter              # Import Tkinter for GUI creation
import random                          # Import random module for shuffling

# ================= GOAL STATE =================

GOAL_STATE = (                         # Define the goal (solved) puzzle state
    1, 2, 3,                           # First row
    4, 5, 6,                           # Second row
    7, 8, 0                            # Third row (0 represents the empty tile)
)

# ================= INDEX / POSITION MAPS =================

INDEX_TO_POSITION = {                 # Map 1D index → (row, column)
    index: (index // 3, index % 3)     # Row = index ÷ 3, Column = index mod 3
    for index in range(9)              # For all 9 positions
}

POSITION_TO_INDEX = {                 # Map (row, column) → 1D index
    (row, col): row * 3 + col          # Convert 2D position back to 1D index
    for row in range(3)                # For each row
    for col in range(3)                # For each column
}

# ================= NEIGHBOR POSITIONS =================

NEIGHBOR_POSITIONS = {}                # Dictionary to store valid moves for each index

for index, (row, col) in INDEX_TO_POSITION.items():  # Loop over each tile position
    neighbors = []                     # List of valid neighbor indexes

    for dr, dc in [(1,0), (-1,0), (0,1), (0,-1)]:  # Possible moves: down, up, right, left
        nr, nc = row + dr, col + dc    # Compute new row and column

        if 0 <= nr < 3 and 0 <= nc < 3: # Check if new position is inside board
            neighbors.append(          # If valid, add neighbor
                POSITION_TO_INDEX[(nr, nc)]
            )

    NEIGHBOR_POSITIONS[index] = neighbors  # Save neighbors for this index



## Heuristic and Puzzle Utilities

In [ ]:
# ================= HEURISTIC FUNCTION =================

def Heuristic_distance_sum(state):     # Heuristic: estimate distance to goal
    distance = 0                       # Initialize total distance

    for index, tile in enumerate(state):  # Loop through each tile
        if tile == 0:                  # Skip the empty tile
            continue

        r1, c1 = INDEX_TO_POSITION[index]      # Current tile position
        goal_index = GOAL_STATE.index(tile)    # Find tile's goal index
        r2, c2 = INDEX_TO_POSITION[goal_index] # Goal position of tile

        distance += abs(r1 - r2) + abs(c1 - c2) # Add Manhattan distance

    return distance                    # Return heuristic value

# ================= SOLVABILITY CHECK =================

def is_solvable(state):                # Check if puzzle is solvable
    tiles = [t for t in state if t != 0]  # Remove empty tile
    inversions = 0                     # Initialize inversion count

    for i in range(len(tiles)):        # Compare every tile
        for j in range(i + 1, len(tiles)):
            if tiles[i] > tiles[j]:    # Count incorrect order pairs
                inversions += 1

    return inversions % 2 == 0          # Solvable if inversions are even

def generate_random_solvable_state():  # Generate a random valid puzzle
    while True:                        # Repeat until valid
        tiles = list(range(9))         # Create tiles 0–8
        random.shuffle(tiles)          # Shuffle randomly
        state = tuple(tiles)           # Convert list to tuple

        if is_solvable(state) and state != GOAL_STATE:  # Check validity
            return state               # Return valid state

# ================= SUCCESSOR FUNCTION =================

def generate_next_states(state):       # Generate all next states from current
    blank_index = state.index(0)       # Find index of empty tile
    next_states = []                   # List to store successors

    for neighbor in NEIGHBOR_POSITIONS[blank_index]:  # For each valid move
        new_state = list(state)        # Convert tuple to list
        new_state[blank_index], new_state[neighbor] = \
            new_state[neighbor], new_state[blank_index]  # Swap tiles
        next_states.append(tuple(new_state))  # Save new state

    return next_states                 # Return successors



## A* Search Algorithm

In [ ]:
# ================= EDUCATIONAL A* SEARCH =================

def a_star_search(start_state):        # A* algorithm (educational version)

    open_list = [[(start_state, 0)]]   # List of paths, each path starts at root
    visited = []                       # List of visited states

    while open_list:                   # While there are paths to explore

        open_list.sort(key=total_cost) # Sort paths by f(n) = g(n) + h(n)

        current_path = open_list.pop(0)  # Remove best path (FIFO)
        current_state = current_path[-1][0]  # Get last state in path

        if current_state in visited:   # Skip already visited states
            continue

        visited.append(current_state)  # Mark state as visited

        if current_state == GOAL_STATE:  # Goal test
            return (
                [state for state, _ in current_path],  # Extract solution path
                visited,                               # Return visited states
                sum(cost for _, cost in current_path)  # Return total cost
            )

        for next_state in generate_next_states(current_state):  # Expand neighbors
            new_path = current_path.copy()   # Copy current path
            new_path.append((next_state, 1)) # Add successor with cost 1
            open_list.append(new_path)       # Add new path to open list

    raise Exception("Search Failed!")   # No solution found

def total_cost(path):                  # Compute f(n) for a path
    g_cost = sum(cost for _, cost in path)  # Sum of path costs (g)
    h_cost = Heuristic_distance_sum(path[-1][0])  # Heuristic cost (h)
    return g_cost + h_cost, path[-1][0]  # Return f(n) + tie-breaker



## Graphical User Interface (Tkinter)

In [ ]:
# ================= GUI CLASS =================

class GUI:
    def __init__(self, window):         # Constructor
        self.window = window            # Store window reference
        self.window.title("8-Puzzle (Educational A*)")  # Set title

        self.current_state = generate_random_solvable_state()  # Initial state
        self.solution_path = []         # List of solution states
        self.current_step = 0           # Animation step index

        self.grid_frame = tkinter.Frame(window, padx=10, pady=10)  # Grid frame
        self.grid_frame.pack()          # Display frame

        self.buttons = []               # Store buttons

        for r in range(3):              # Create 3 rows
            row_buttons = []
            for c in range(3):          # Create 3 columns
                btn = tkinter.Button(
                    self.grid_frame,
                    width=6,
                    height=3,
                    font=("Segoe UI", 16, "bold"),
                    command=lambda r=r, c=c: self.tile_clicked(r, c)
                )
                btn.grid(row=r, column=c, padx=5, pady=5)  # Place button
                row_buttons.append(btn)
            self.buttons.append(row_buttons)

        control = tkinter.Frame(window) # Control bar
        control.pack(fill="x")

        self.status = tkinter.StringVar()  # Status text
        self.status.set("Ready")

        tkinter.Label(control, textvariable=self.status).pack(side="left")
        tkinter.Button(control, text="Shuffle", command=self.shuffle).pack(side="right")
        tkinter.Button(control, text="Solve (A*)", command=self.solve).pack(side="right")

        self.render()                   # Draw initial state

    def render(self):                  # Update GUI
        for index, tile in enumerate(self.current_state):
            r, c = INDEX_TO_POSITION[index]
            btn = self.buttons[r][c]

            if tile == 0:
                btn.config(text="", bg="#dddddd", relief="sunken")
            else:
                btn.config(text=str(tile), bg="#ffffff", relief="raised")

        if self.current_state == GOAL_STATE:
            self.status.set("Solved ✔")
        else:
            self.status.set(
                f"Heuristic Distance: {Heuristic_distance_sum(self.current_state)}"
            )

    def tile_clicked(self, r, c):       # Handle tile click
        tile_index = POSITION_TO_INDEX[(r, c)]
        blank_index = self.current_state.index(0)

        if tile_index in NEIGHBOR_POSITIONS[blank_index]:
            state = list(self.current_state)
            state[blank_index], state[tile_index] = \
                state[tile_index], state[blank_index]
            self.current_state = tuple(state)
            self.render()

    def shuffle(self):                 # Shuffle puzzle
        self.current_state = generate_random_solvable_state()
        self.solution_path = []
        self.render()

    def solve(self):                   # Solve puzzle using A*
        self.solution_path, _, _ = a_star_search(self.current_state)
        self.current_step = 0
        self.animate()

    def animate(self, delay=300):      # Animate solution
        if self.current_step >= len(self.solution_path):
            self.status.set("Solved ✔")
            return

        self.current_state = self.solution_path[self.current_step]
        self.current_step += 1
        self.render()
        self.window.after(delay, self.animate)



## Main Entry Point

In [ ]:
# ================= MAIN =================

def main():                            # Program entry point
    window = tkinter.Tk()              # Create main window
    GUI(window)             # Create GUI
    window.mainloop()                  # Start event loop

if __name__ == "__main__":
    main()                             # Run program